# GP + BO crutch personalisation -- function-by-function validation

This notebook exercises every public function in `src/optimization/` against the real `data_V2/` data. Each cell focuses on one piece so you can visually confirm shapes, sanity values and figures.

Pipeline outline:

1. `data.load_joined_dataset` -> joined DataFrame (one row per metabolics trial).
2. `data.build_candidate_grid` -> dense 140-combo BO candidate set.
3. `gp.fit_gp_hyperparameters` -> fit a GP on a single participant; plot 1D slice.
4. `pipeline.suggest_next_for` -> live next-geometry suggestion in 3 prior modes.
5. `pipeline.run_personalised_bo` -> simulated BO replay in 3 prior modes.
6. `viz.plot_outcome_per_geometry` -> mean+std bar across 10 participants.
7. `viz.plot_correlation_matrix` -> input x output Pearson heatmap.


In [1]:
"""Cell 0 - imports + path setup."""
from __future__ import annotations
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from optimization import data as opt_data
from optimization import gp as opt_gp
from optimization import bo as opt_bo
from optimization import pipeline as opt_pipeline
from optimization import viz as opt_viz

DATA_ROOT = REPO_ROOT / "data_V2"
TARGET_MIH = "MIH19"
RANDOM_SEED = 0
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())


REPO_ROOT: /Users/riccardoconci/Local_documents/!CURRENT_RESEARCH/Personalising-Crutches
DATA_ROOT exists: True


## Cell 1 - load + join the three JSON sources

In [2]:
df = opt_data.load_joined_dataset(DATA_ROOT)
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print()
print("rows per participant:")
print(df.groupby("participant").size())
print()
print("NaNs per column:")
print(df.isna().sum())
df.head()


shape: (140, 17)
columns: ['participant', 'trial_key', 'alpha', 'beta', 'gamma', 'height', 'weight', 'forearm_length', 'age', 'activity_level', 'sex_male', 'prev_crutch', 'cot_linear', 'cot_raw', 'sus', 'nrs', 'tlx']

rows per participant:
participant
MIH01    14
MIH02    14
MIH15    14
MIH16    14
MIH19    14
MIH27    14
MIH29    14
MIH32    14
MIH33    14
MIH35    14
dtype: int64

NaNs per column:
participant       0
trial_key         0
alpha             0
beta              0
gamma             0
height            0
weight            0
forearm_length    0
age               0
activity_level    0
sex_male          0
prev_crutch       0
cot_linear        0
cot_raw           0
sus               4
nrs               4
tlx               4
dtype: int64


,participant,trial_key,alpha,beta,gamma,height,weight,forearm_length,age,activity_level,sex_male,prev_crutch,cot_linear,cot_raw,sus,nrs,tlx
0,MIH01,baseline_1,95.0,95.0,0.0,175.0,70.3,20.0,25.0,4.0,1.0,0.0,11.418353,9.847382,72.916667,0.0,41.5
1,MIH01,1,105.0,95.0,0.0,175.0,70.3,20.0,25.0,4.0,1.0,0.0,13.499527,12.170244,62.500000,0.0,54.0
2,MIH01,2,105.0,125.0,0.0,175.0,70.3,20.0,25.0,4.0,1.0,0.0,13.785837,12.698241,54.166667,1.0,57.0
3,MIH01,3,85.0,125.0,0.0,175.0,70.3,20.0,25.0,4.0,1.0,0.0,13.006126,12.160218,70.833333,0.0,43.0
4,MIH01,4,85.0,95.0,0.0,175.0,70.3,20.0,25.0,4.0,1.0,0.0,17.028378,16.424158,45.833333,1.0,52.0


## Cell 2 - dense 140-combo candidate grid

In [3]:
grid = opt_data.build_candidate_grid()
print("grid shape:", grid.shape, "(should be (140, 3))")
print("alpha values:", sorted(grid["alpha"].unique()))
print("beta  values:", sorted(grid["beta"].unique()))
print("gamma values:", sorted(grid["gamma"].unique()))
grid.head()


grid shape: (140, 3) (should be (140, 3))
alpha values: [np.float64(85.0), np.float64(92.5), np.float64(100.0), np.float64(105.0)]
beta  values: [np.float64(95.0), np.float64(102.5), np.float64(110.0), np.float64(117.5), np.float64(125.0)]
gamma values: [np.float64(-9.0), np.float64(-6.0), np.float64(-3.0), np.float64(0.0), np.float64(3.0), np.float64(6.0), np.float64(9.0)]


,alpha,beta,gamma
0,85.0,95.0,-9.0
1,85.0,95.0,-6.0
2,85.0,95.0,-3.0
3,85.0,95.0,0.0
4,85.0,95.0,3.0


## Cell 3 - fit a GP on a single participant; visualise 1D alpha sweep

We z-score features using stats fit on `MIH19`'s rows alone, fit a Matern-5/2 GP on the combined objective, then sweep `alpha` while holding the other geometry coords at zero (z-scored midpoint).

In [4]:
single_df = df[df.participant == TARGET_MIH].copy()
bundle = opt_data.make_training_bundle(single_df, objective="combined")
model = opt_gp.fit_gp_hyperparameters(bundle.X_train, bundle.y_train, kernel="matern52", n_restarts=3, seed=RANDOM_SEED)
print("fitted theta:", np.round(model.theta, 4))
print("NLL:", round(model.train_nll, 3))

alpha_grid = np.linspace(85, 105, 60)
alpha_z_grid = (alpha_grid - bundle.feature_stats.means['alpha']) / max(bundle.feature_stats.stds['alpha'], 1e-9)
feature_means = np.array([bundle.feature_stats.means[c] for c in opt_data.INPUT_FEATURES])
feature_stds = np.array([max(bundle.feature_stats.stds[c], 1e-9) for c in opt_data.INPUT_FEATURES])
x_star = np.tile((feature_means - feature_means) / feature_stds, (alpha_grid.size, 1))
alpha_idx = list(opt_data.INPUT_FEATURES).index('alpha')
x_star[:, alpha_idx] = alpha_z_grid
post = model.predict(x_star)
lo, hi = post.mean - 1.96 * np.sqrt(post.var), post.mean + 1.96 * np.sqrt(post.var)

fig = go.Figure()
fig.add_trace(go.Scatter(x=alpha_grid, y=post.mean, mode='lines', name='GP mean', line=dict(color='royalblue')))
fig.add_trace(go.Scatter(x=np.concatenate([alpha_grid, alpha_grid[::-1]]),
                          y=np.concatenate([hi, lo[::-1]]), fill='toself',
                          fillcolor='rgba(65,105,225,0.18)', line=dict(width=0),
                          showlegend=False, name='95% CI'))
obs_alpha = single_df['alpha']
obs_y = bundle.y_train[:len(obs_alpha)]
fig.add_trace(go.Scatter(x=obs_alpha, y=obs_y[:len(obs_alpha)], mode='markers',
                          marker=dict(size=10, color='black'), name='observations'))
fig.update_layout(title=f"{TARGET_MIH}: GP posterior over alpha (other features at MIH19 mean)",
                   xaxis_title='alpha (deg)', yaxis_title='z-scored composite objective',
                   width=820, height=480)
fig.show()


fitted theta: [9.0641e+00 1.0000e-03 1.0000e-03]
NLL: 30.587


## Cell 4 - `suggest_next_for` in all 3 modes

`POOL_INCLUDING_SELF` and `POOL_LEAVE_ONE_OUT` train a GP on the chosen prior pool; `NO_PRIOR` requires target-side seed observations (handled in the simulated-BO cell below).

In [5]:
for mode in [opt_pipeline.PriorMode.POOL_INCLUDING_SELF, opt_pipeline.PriorMode.POOL_LEAVE_ONE_OUT]:
    sug = opt_pipeline.suggest_next_for(
        TARGET_MIH, df=df, mode=mode, objective='combined', seed=RANDOM_SEED,
    )
    print(f"[{mode.value:>16}] next: alpha={sug.next_alpha}, beta={sug.next_beta}, gamma={sug.next_gamma}, alpha_value={sug.acquisition_value:.4f}")
    print(f"                   posterior mean range: [{sug.posterior_mean.min():.3f}, {sug.posterior_mean.max():.3f}]")
    print(f"                   posterior var  range: [{sug.posterior_var.min():.3f}, {sug.posterior_var.max():.3f}]")

print()
print("NO_PRIOR mode for live suggestions: callers should pass df=df[df.participant==TARGET] with mode=POOL_INCLUDING_SELF")
print("to use only the target's own data. Demonstrated in the simulated-BO cell.")


[  pool_with_self] next: alpha=92.5, beta=95.0, gamma=-9.0, alpha_value=0.3251
                   posterior mean range: [-4.723, -1.036]
                   posterior var  range: [0.001, 2.325]
[        pool_loo] next: alpha=105.0, beta=95.0, gamma=-9.0, alpha_value=0.0049
                   posterior mean range: [-0.101, 0.132]
                   posterior var  range: [3.893, 3.894]

NO_PRIOR mode for live suggestions: callers should pass df=df[df.participant==TARGET] with mode=POOL_INCLUDING_SELF
to use only the target's own data. Demonstrated in the simulated-BO cell.


## Cell 5 - simulated BO in all 3 prior modes

Each mode replays the BO loop on `MIH19`'s 14 measured geometries. We compare the best-y curves to see how prior data accelerates convergence. Lower y is better.

In [6]:
results = {}
for mode in opt_pipeline.PriorMode:
    res = opt_pipeline.run_personalised_bo(
        TARGET_MIH, df=df, mode=mode, objective='combined',
        n_iter=8, seed_n=2, kernel='matern52', acquisition='EI',
        n_restarts=3, seed=RANDOM_SEED,
    )
    results[mode] = res
    print(f"[{mode.value:>16}] best_y={res.best_y:.3f} at geom={res.best_geometry}, n_visited={len(res.history.visited_indices)}")

fig = go.Figure()
for mode, res in results.items():
    fig.add_trace(go.Scatter(
        x=list(range(len(res.history.best_y_curve))),
        y=res.history.best_y_curve,
        mode='lines+markers', name=mode.value,
    ))
global_min = float(results[opt_pipeline.PriorMode.POOL_INCLUDING_SELF].target_geometries['y_objective'].min())
fig.add_hline(y=global_min, line_dash='dash', line_color='gray',
              annotation_text=f'global min = {global_min:.3f}', annotation_position='bottom right')
fig.update_layout(title=f'{TARGET_MIH}: BO best-y vs iteration (3 modes)',
                   xaxis_title='iteration (incl. seed)', yaxis_title='best y (lower = better)',
                   width=820, height=460)
fig.show()


[  pool_with_self] best_y=-4.745 at geom=(85.0, 95.0, -9.0), n_visited=8
[        pool_loo] best_y=-5.368 at geom=(85.0, 95.0, -9.0), n_visited=10
[        no_prior] best_y=-4.414 at geom=(85.0, 95.0, -9.0), n_visited=10


## Cell 6 - outcome per geometry (mean ± std across 10 participants)

In [7]:
for outcome in ['cot_linear', 'survey', 'combined']:
    fig = opt_viz.plot_outcome_per_geometry(df, outcome=outcome, title_prefix='gridsearch (n=10): ')
    fig.show()


## Cell 7 - input x output correlation matrix

In [8]:
fig = opt_viz.plot_correlation_matrix(df, method='pearson', title_prefix='gridsearch (n=10): ')
fig.show()

fig = opt_viz.plot_correlation_matrix(df, method='spearman', title_prefix='gridsearch (n=10): ')
fig.show()
